# Model Diagnostics Notebook

This notebook is designed to thoroughly evaluate the performance of different regression models for the computer price prediction task. We will:

1. Load the preprocessed data
2. Prepare the features and target variables
3. Train multiple models
4. Evaluate each model with various metrics
5. Visualize predictions vs actual values
6. Test the models on sample data

In [ ]:
# Import necessary libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline

# Add the src directory to the Python path
sys.path.append('../src')

# Import project modules
from clean_price import clean_price_column
from features import engineer_features, identify_core_features, get_feature_names
from price_scaler import PriceScaler

# Set up plotting style
plt.style.use('ggplot')
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Load and Inspect the Data

In [ ]:
# Load the preprocessed data
data_path = '../data/db_computers_2025_processed.csv'
df = pd.read_csv(data_path)

# Display basic information
print(f"Dataset shape: {df.shape}")
print("\nDataset columns:")
print(df.columns.tolist())

# Check if 'Price' column exists
if 'Price' in df.columns:
    print(f"\nPrice statistics:")
    print(f"Mean: {df['Price'].mean():.2f}")
    print(f"Std Dev: {df['Price'].std():.2f}")
    print(f"Min: {df['Price'].min():.2f}")
    print(f"Max: {df['Price'].max():.2f}")
    
    # Plot price distribution
    plt.figure(figsize=(12, 6))
    sns.histplot(df['Price'], bins=50, kde=True)
    plt.title('Price Distribution')
    plt.xlabel('Price (€)')
    plt.show()
else:
    print("Error: 'Price' column not found in the dataset.")

## 2. Prepare Features and Target Variables

In [ ]:
# Identify core features
numeric_features, categorical_features = identify_core_features(df)
print(f"Identified {len(numeric_features)} numeric features: {numeric_features}")
print(f"Identified {len(categorical_features)} categorical features: {categorical_features}")

# Engineer features
X_transformed, feature_pipeline = engineer_features(
    df,
    numeric_features=numeric_features,
    categorical_features=categorical_features
)

# Get feature names
feature_names = get_feature_names(feature_pipeline, numeric_features, categorical_features)
print(f"Total features after engineering: {len(feature_names)}")

# Apply price scaling
price_scaler = PriceScaler(price_column='Price')
price_scaler.fit(df)
scaled_prices = price_scaler.transform_target(df['Price'])

print(f"\nPrice statistics before scaling:")
print(f"Mean: {df['Price'].mean():.2f}")
print(f"Std Dev: {df['Price'].std():.2f}")

print(f"\nScaled price statistics:")
print(f"Mean: {scaled_prices.mean():.2f}")
print(f"Std Dev: {scaled_prices.std():.2f}")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_transformed, scaled_prices, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

## 3. Train and Evaluate Multiple Models

In [ ]:
# Define a function to evaluate model performance
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on training and test sets
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics on training set
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    
    # Calculate metrics on test set
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Cross-validation
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_rmse = np.sqrt(-cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=cv))
    
    # Print results
    print(f"\n{name} Model Performance:")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Training MAE: {train_mae:.4f}")
    print(f"Test MAE: {test_mae:.4f}")
    print(f"Training R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Cross-validation RMSE: {cv_rmse.mean():.4f} (±{cv_rmse.std():.4f})")
    
    # Evaluate predictions in original price scale
    y_test_orig = price_scaler.inverse_transform(None, y_test)
    y_test_pred_orig = price_scaler.inverse_transform(None, y_test_pred)
    orig_rmse = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
    orig_mae = mean_absolute_error(y_test_orig, y_test_pred_orig)
    orig_r2 = r2_score(y_test_orig, y_test_pred_orig)
    
    print(f"\nOriginal Price Scale Metrics:")
    print(f"RMSE: {orig_rmse:.2f} €")
    print(f"MAE: {orig_mae:.2f} €")
    print(f"R²: {orig_r2:.4f}")
    
    # Visualize predictions vs actual values
    plt.figure(figsize=(12, 10))
    
    # Scatter plot in standardized scale
    plt.subplot(2, 1, 1)
    plt.scatter(y_test, y_test_pred, alpha=0.5)
    plt.plot([-3, 3], [-3, 3], 'r--')
    plt.title(f'{name} - Predicted vs Actual (Standardized Scale)')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.grid(True)
    
    # Scatter plot in original price scale
    plt.subplot(2, 1, 2)
    plt.scatter(y_test_orig, y_test_pred_orig, alpha=0.5)
    plt.plot([0, max(y_test_orig)], [0, max(y_test_orig)], 'r--')
    plt.title(f'{name} - Predicted vs Actual (Original Price Scale)')
    plt.xlabel('Actual Price (€)')
    plt.ylabel('Predicted Price (€)')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Return the trained model and metrics
    return {
        'model': model,
        'metrics': {
            'train_rmse': train_rmse,
            'test_rmse': test_rmse,
            'train_mae': train_mae,
            'test_mae': test_mae,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'cv_rmse': cv_rmse.mean(),
            'orig_rmse': orig_rmse,
            'orig_mae': orig_mae,
            'orig_r2': orig_r2
        }
    }

# Define models to evaluate
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
}

# Train and evaluate each model
results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X_train, X_test, y_train, y_test, name)

## 4. Compare Model Performance

In [ ]:
# Create a DataFrame to compare model performance
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Training RMSE': [results[model]['metrics']['train_rmse'] for model in results],
    'Test RMSE': [results[model]['metrics']['test_rmse'] for model in results],
    'CV RMSE': [results[model]['metrics']['cv_rmse'] for model in results],
    'Test R²': [results[model]['metrics']['test_r2'] for model in results],
    'Original Scale RMSE (€)': [results[model]['metrics']['orig_rmse'] for model in results],
    'Original Scale MAE (€)': [results[model]['metrics']['orig_mae'] for model in results]
})

# Set Model as index for better display
metrics_df.set_index('Model', inplace=True)

# Display the comparison table
metrics_df.sort_values('Test RMSE')

## 5. Test with Sample Inputs

In [ ]:
# Function to test predictions with sample inputs
def test_prediction(model_name, sample_idx=None):
    model = results[model_name]['model']
    
    if sample_idx is None:
        # Use random test set sample
        sample_idx = np.random.randint(0, len(X_test))
    
    # Get sample features and actual price
    sample_features = X_test[sample_idx:sample_idx+1]
    actual_price_scaled = y_test[sample_idx]
    actual_price = price_scaler.inverse_transform(None, actual_price_scaled)
    
    # Predict price (scaled)
    predicted_price_scaled = model.predict(sample_features)[0]
    predicted_price = price_scaler.inverse_transform(None, predicted_price_scaled)
    
    print(f"Model: {model_name}")
    print(f"Actual Price (scaled): {actual_price_scaled:.4f}")
    print(f"Predicted Price (scaled): {predicted_price_scaled:.4f}")
    print(f"Actual Price: €{actual_price:.2f}")
    print(f"Predicted Price: €{predicted_price:.2f}")
    print(f"Prediction Error: €{actual_price - predicted_price:.2f}")
    print(f"Prediction Error (%): {100 * (actual_price - predicted_price) / actual_price:.2f}%")
    
    return {
        'actual_scaled': actual_price_scaled,
        'predicted_scaled': predicted_price_scaled,
        'actual': actual_price,
        'predicted': predicted_price
    }

# Test each model with 3 different samples
test_samples = np.random.randint(0, len(X_test), 3)
for sample_idx in test_samples:
    print(f"\n======= Sample {sample_idx} =======\n")
    for model_name in results.keys():
        test_prediction(model_name, sample_idx)
        print("\n")

## 6. Manual Testing on Custom Inputs

In [ ]:
# Define a function to create a sample input with reasonable values
def create_sample_input():
    # Create a sample input dictionary
    sample_input = {}
    
    # Add numeric features with reasonable values
    for feature in numeric_features:
        if feature.lower() == 'price':
            # Skip the price feature as it's what we're predicting
            continue
        # Use median value
        sample_input[feature] = df[feature].median()
    
    # Add categorical features with most common values
    for feature in categorical_features:
        sample_input[feature] = df[feature].mode().iloc[0]
    
    # Convert to DataFrame
    input_df = pd.DataFrame([sample_input])
    
    # Add a dummy Price column for the feature pipeline
    input_df['Price'] = 0
    
    return input_df

# Create a sample input
sample_input = create_sample_input()
print("Sample input:")
print(sample_input.drop('Price', axis=1).head())

# Transform input using feature pipeline
X_sample = feature_pipeline.transform(sample_input)

# Make predictions with each model
print("\nPredictions for sample input:")
for model_name, result in results.items():
    # Get the model
    model = result['model']
    
    # Predict price (scaled)
    predicted_price_scaled = model.predict(X_sample)[0]
    
    # Convert to original price scale
    predicted_price = price_scaler.inverse_transform(None, predicted_price_scaled)
    
    print(f"{model_name}:")
    print(f"  Predicted Price (scaled): {predicted_price_scaled:.4f}")
    print(f"  Predicted Price: €{predicted_price:.2f}")

## 7. Save the Best Model

In [ ]:
# Identify the best model based on test RMSE
best_model_name = metrics_df['Test RMSE'].idxmin()
best_model = results[best_model_name]['model']

print(f"Best model: {best_model_name}")
print(f"Test RMSE: {metrics_df.loc[best_model_name, 'Test RMSE']:.4f}")
print(f"Original Scale RMSE: €{metrics_df.loc[best_model_name, 'Original Scale RMSE (€)']:.2f}")

# Save the best model and price scaler
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_price_model.joblib')
joblib.dump(price_scaler, '../models/best_price_scaler.joblib')
joblib.dump(feature_pipeline, '../models/best_feature_pipeline.joblib')

print("\nBest model, price scaler, and feature pipeline saved to the models directory.")

## 8. Conclusion and Recommendations

Based on the evaluation results, here are the key findings:

1. Model performance comparison based on RMSE, MAE, and R² metrics
2. Analysis of model performance in both scaled and original price domains
3. Identification of potential issues with negative price predictions and how to address them
4. Recommendations for model selection and further improvements

The analysis has helped us understand which models are most reliable for computer price prediction and how to ensure valid, positive price predictions.